In [ ]:
# --- Importaciones ---------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay,
)

import sklearn
print(sklearn.__version__)  # anota la versión

In [ ]:
# --- Carga de MNIST desde Keras --------------------------------------
import tensorflow as tf

# Cargar el dataset MNIST
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

# Concatenar conjuntos de entrenamiento y prueba para tener un único conjunto X_mnist, y_mnist
X_mnist_raw = np.concatenate([X_train_full, X_test], axis=0)
y_mnist_raw = np.concatenate([y_train_full, y_test], axis=0)

# Normalizar los datos (escalar a 0-1) y aplanar las imágenes de 28x28 a 784 características
X_mnist = X_mnist_raw.astype('float32') / 255.0
X_mnist = X_mnist.reshape(-1, 28 * 28) # Aplanar a 784 características
y_mnist = y_mnist_raw.astype(int)

print(f'Dataset completo: {X_mnist.shape}')  # (70000, 784)
print(f'Clases: {np.unique(y_mnist)}')        # [0 1 2 3 4 5 6 7 8 9]

# --- Subconjunto de 10,000 muestras para Colab -----------------------
semilla = 42
rng = np.random.default_rng(semilla)
indices = rng.choice(len(X_mnist), size=10_000, replace=False)
X = X_mnist[indices]
y = y_mnist[indices]

print(f'Subconjunto: {X.shape}')

In [ ]:
# --- Visualización de un dígito de ejemplo ----------------------------

# Selecciona un índice aleatorio para mostrar una imagen de ejemplo
ejemplo_idx = rng.integers(0, len(X))

# Obtiene la imagen y la etiqueta correspondientes
imagen_ejemplo = X[ejemplo_idx]
etiqueta_ejemplo = y[ejemplo_idx]

# Reshape la imagen de 784 características a una matriz de 28x28
# Si X_mnist_raw es 3D (num_images, height, width), no necesitarías reshapear
# pero como X se aplanó a (10000, 784), necesitamos revertir eso.
imagen_2d = imagen_ejemplo.reshape(28, 28)

# Muestra la imagen usando matplotlib
plt.figure(figsize=(4, 4))
plt.imshow(imagen_2d, cmap='binary') # 'binary' para dígitos en blanco y negro
plt.title(f'Dígito: {etiqueta_ejemplo}')
plt.axis('off') # Oculta los ejes para una mejor visualización
plt.show()

In [ ]:
# --- Split de datos --------------------------------------------------
# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=semilla, stratify=y
)

print(f'Forma de X_train: {X_train.shape}') # (8000, 784)
print(f'Forma de X_test: {X_test.shape}')   # (2000, 784)

# --- Pipeline con escalado -------------------------------------------
# Crear un pipeline que primero escala los datos y luego puede tener un modelo
# Aquí solo definimos el escalador, el modelo se añadirá más tarde.
pipeline_scaler = Pipeline([
    ('scaler', StandardScaler())
])

# Aplicar el pipeline al conjunto de entrenamiento y prueba
# Esto es para obtener los datos escalados para posibles análisis o modelos directos
X_train_scaled = pipeline_scaler.fit_transform(X_train)
X_test_scaled = pipeline_scaler.transform(X_test)

print(f'Forma de X_train_scaled: {X_train_scaled.shape}')
print(f'Forma de X_test_scaled: {X_test_scaled.shape}')

In [ ]:
# --- Entrenar y evaluar KNN (k=5) ------------------------------------

# Crear una instancia del clasificador KNN con k=5
knn_clf = KNeighborsClassifier(n_neighbors=5)

# Entrenar el modelo con los datos de entrenamiento escalados
start_time = time.time()
knn_clf.fit(X_train_scaled, y_train)
end_time = time.time()
t_fit_knn = end_time - start_time
print(f'Tiempo de entrenamiento KNN: {t_fit_knn:.2f} segundos')

# Realizar predicciones sobre los datos de prueba escalados
start_time = time.time()
y_pred_knn = knn_clf.predict(X_test_scaled)
end_time = time.time()
t_pred_knn = end_time - start_time
print(f'Tiempo de predicción KNN: {t_pred_knn:.2f} segundos')

# Evaluar el rendimiento del modelo
accuracy_knn = accuracy_score(y_test, y_pred_knn)
print(f'Precisión del modelo KNN (k=5): {accuracy_knn:.4f}')

# Mostrar el informe de clasificación
print('\nInforme de Clasificación KNN (k=5):\n')
print(classification_report(y_test, y_pred_knn))

# Mostrar la matriz de confusión
print('\nMatriz de Confusión KNN (k=5):\n')
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay.from_estimator(knn_clf, X_test_scaled, y_test, cmap=plt.cm.Blues, ax=ax)
plt.title('Matriz de Confusión KNN (k=5)')
plt.show()

In [ ]:
# --- Comparar distintos valores de k --------------------------------
valores_k = [1, 3, 5, 7, 10, 15, 20]
acc_por_k = []

for k in valores_k:
    knn_k = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    knn_k.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn_k.predict(X_test_scaled))
    acc_por_k.append(acc)
    print(f'k={k:2d} | Acc: {acc:.4f}')

In [ ]:
# --- SVC con kernel RBF ---------------------------------------------
t0 = time.time()
svc = SVC(kernel='rbf', C=5.0, gamma='scale', random_state=semilla)
svc.fit(X_train_scaled, y_train)
t_fit_svc = time.time() - t0

t0 = time.time()
y_pred_svc = svc.predict(X_test_scaled)
t_pred_svc = time.time() - t0

acc_svc = accuracy_score(y_test, y_pred_svc)
print(f'SVC(rbf) | Acc: {acc_svc:.4f}',
    f'| Fit: {t_fit_svc:.1f}s | Pred: {t_pred_svc:.2f}s')


In [ ]:
# --- Explorar C y gamma (tabla reducida) ----------------------------
# Referencia: gamma='scale' calcula γ ≈ 0.021 para MNIST normalizado.
# Valores muy por debajo (0.001) → subajuste; muy por encima (0.1) → sobreajuste.
resultados_svc = []

for C in [0.1, 1.0, 5.0, 10.0]:
    for gm in ['scale', 0.01, 0.001]:
        modelo = SVC(kernel='rbf', C=C, gamma=gm,
                    random_state=semilla)
        modelo.fit(X_train_scaled, y_train)
        acc = accuracy_score(y_test, modelo.predict(X_test_scaled))
        resultados_svc.append({'C': C, 'gamma': gm, 'acc': acc})
        print(f'C={C:5.1f} gamma={str(gm):7s} | Acc={acc:.4f}')

In [ ]:
# --- Dígitos mal clasificados por SVC -------------------------------
errores = np.where(y_pred_svc != y_test)[0]
print(f'Total errores: {len(errores)} de {len(y_test)}')

fig, ejes = plt.subplots(3, 8, figsize=(14, 6))
for i, idx_err in enumerate(errores[:24]):
    ax = ejes[i // 8, i % 8]
    ax.imshow(X_test_scaled[idx_err].reshape(28, 28), cmap='gray_r')
    ax.set_title(
        f'Real:{y_test[idx_err]}\nPred:{y_pred_svc[idx_err]}',
        fontsize=7,
    )
    ax.axis('off')

plt.suptitle('Ejemplos mal clasificados por SVC', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Random Forest como referencia ----------------------------------
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=100, random_state=semilla, n_jobs=-1
)
rf.fit(X_train_scaled, y_train)
t_fit_rf = time.time() - t0

t0 = time.time()
acc_rf = accuracy_score(y_test, rf.predict(X_test_scaled))
t_pred_rf = time.time() - t0

print('Modelo       | Fit(s) | Pred(s) | Accuracy')
print(f'KNN(k=5)     | {t_fit_knn:6.2f} | {t_pred_knn:7.2f} | {accuracy_knn:.4f}')
print(f'SVC(rbf,C=5) | {t_fit_svc:6.1f} | {t_pred_svc:7.2f} | {acc_svc:.4f}')
print(f'RandomForest | {t_fit_rf:6.2f} | {t_pred_rf:7.2f} | {acc_rf:.4f}')

In [ ]:
# --- Matriz de confusión del mejor modelo (SVC) ---------------------
fig, ax = plt.subplots(figsize=(9, 7))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_svc,
    display_labels=list(range(10)),
    cmap='Blues',
    ax=ax,
)
ax.set_title('Matriz de confusión — SVC (kernel RBF, C=5)')
plt.tight_layout()
plt.show()

# Informe completo
print(classification_report(y_test, y_pred_svc))